# 02 - Data Cleaning

## Customer Intelligence Platform

---
- This notebook executes the data cleaning pipeline defined in `src/cleaning.py`.  
- The pipeline removes leakage columns, fixes data types, handles missing values,
and produces a clean dataset for downstream analysis.


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

# imports from self defined module
from src.load_data import load_raw
from src.cleaning import clean_data, run_cleaning_pipeline
from src.config import CLEAN_FILE, TARGET
from src.utils import df_summary


## Load Raw Data

---

In [2]:
df_raw = load_raw()
df_summary(df_raw, "Raw Data")

✅ Loaded raw data: 7,043 rows × 33 cols

[INFO] Raw Data
   Shape: 7,043 rows x 33 columns

Column Types:
 str        24
int64       6
float64     3
Name: count, dtype: int64

   Memory: 2.93 MB
   Missing values: 5,174


## 2. Run Cleaning Pipeline

---
The cleaning pipeline performs the following steps:
1. Drop 8 leakage/irrelevant columns
2. Drop 5 geographic columns (loaded separately when needed)
3. Convert `Total Charges` from string to numeric
4. Impute 11 missing `Total Charges` values with $0 (new customers with 0 tenure)
5. Verify target variable integrity


In [3]:
df_clean = clean_data(df_raw)
df_summary(df_clean, "Cleaned Data")

   Dropped 8 leakage/irrelevant columns
   Dropped 5 geographic columns (loaded separately)
   Converted Total Charges to numeric (11 coerced to NaN)
   Imputed 11 missing Total Charges with 0.0
   Target 'Churn Value': {0: 5174, 1: 1869}
[TIME] clean_data completed in 0.01s

[INFO] Cleaned Data
   Shape: 7,043 rows x 20 columns

Column Types:
 str        16
int64       2
float64     2
Name: count, dtype: int64

   Memory: 1.70 MB
   Duplicate rows: 22


In [4]:
print(f"Columns retained ({len(df_clean.columns)}):")
for i, col in enumerate(df_clean.columns):
    print(f"  {i+1}. {col} ({df_clean[col].dtype})")


Columns retained (20):
  1. Gender (str)
  2. Senior Citizen (str)
  3. Partner (str)
  4. Dependents (str)
  5. Tenure Months (int64)
  6. Phone Service (str)
  7. Multiple Lines (str)
  8. Internet Service (str)
  9. Online Security (str)
  10. Online Backup (str)
  11. Device Protection (str)
  12. Tech Support (str)
  13. Streaming TV (str)
  14. Streaming Movies (str)
  15. Contract (str)
  16. Paperless Billing (str)
  17. Payment Method (str)
  18. Monthly Charges (float64)
  19. Total Charges (float64)
  20. Churn Value (int64)


## Validation Checks

---

In [5]:
# Check no leakage columns remain
from src.config import LEAKAGE_COLUMNS, GEOGRAPHIC_COLUMNS
leaked = [c for c in LEAKAGE_COLUMNS if c in df_clean.columns]
geo = [c for c in GEOGRAPHIC_COLUMNS if c in df_clean.columns]
print(f"Leakage columns remaining: {leaked or 'None'}")
print(f"Geographic columns remaining: {geo or 'None'}")

# Check Total Charges is numeric
print(f"\nTotal Charges dtype: {df_clean['Total Charges'].dtype}")
print(f"Total Charges nulls: {df_clean['Total Charges'].isnull().sum()}")

# Check target
print(f"\nTarget distribution:")
print(df_clean[TARGET].value_counts())


Leakage columns remaining: None
Geographic columns remaining: None

Total Charges dtype: float64
Total Charges nulls: 0

Target distribution:
Churn Value
0    5174
1    1869
Name: count, dtype: int64


## Save Cleaned Data

---

In [6]:
from src.utils import save_dataframe
save_dataframe(df_clean, CLEAN_FILE)
print(f"Saved to: {CLEAN_FILE}")


[OK] Saved 7,043 rows x 20 cols -> telco_clean.csv
Saved to: S:\customer-intelligence-platform\data\interim\telco_clean.csv


## Summary

| Step | Before | After |
|------|--------|-------|
| Rows | 7,043 | 7,043 |
| Columns | 33 | 20 |
| Missing Values | 5,185 | 0 |
| Data Types Fixed | Total Charges (object -> float64) | Done |

The cleaned dataset is saved to `data/interim/telco_clean.csv` and is ready for
exploratory data analysis and feature engineering.
